# 01: Non-IID FedAvg vs central training

Runs the Step 4 experiment on a free GPU: for each Dirichlet alpha, FedAvg across 10 simulated sites, plus one central baseline per dataset. Logs go to `runs/` and are zipped at the end.

Settings: GPU on (Kaggle: Settings -> Accelerator; Colab: Runtime -> Change runtime type). Run with `SMOKE = True` first.

In [ ]:
import os
import subprocess
import sys

SMOKE = True  # True: 2 rounds on 500 training images, to check the plumbing
REPO = "Nathaniel47/fedsurveil"

token = os.environ.get("GITHUB_TOKEN")
try:  # Kaggle: Add-ons -> Secrets, name it GITHUB_TOKEN (only needed while the repo is private)
    from kaggle_secrets import UserSecretsClient

    token = token or UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("fedsurveil"):
    subprocess.run(["git", "clone", "-q", url, "fedsurveil"], check=True)
os.chdir("fedsurveil")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ml.txt"], check=True
)
import torch

print(torch.__version__, "cuda:", torch.cuda.is_available())
assert torch.cuda.is_available() or SMOKE, "No GPU: set Runtime -> Change runtime type -> T4 GPU, then run again"


In [ ]:
!python -m fedsurveil.data.fetch

In [ ]:
import itertools
import subprocess
import sys

DATASETS = ["bloodmnist", "pathmnist"]
ALPHAS = [0.1, 0.5, 100.0]
SEEDS = [0] if SMOKE else [0, 1, 2]
ROUNDS, LIMIT = (2, 500) if SMOKE else (30, 0)


def go(*args):
    cmd = [sys.executable, "-m", "fedsurveil.fl.simulate", *map(str, args)]
    print(" ".join(cmd[2:]))
    subprocess.run(cmd, check=True)


for ds, seed in itertools.product(DATASETS, SEEDS):
    common = ["--dataset", ds, "--seed", seed, "--rounds", ROUNDS, "--limit", LIMIT or 0]
    go(*common, "--algorithm", "central", "--alpha", 100.0)
    for a in ALPHAS:
        go(*common, "--algorithm", "fedavg", "--alpha", a)

In [ ]:
import glob
import json

for path in sorted(glob.glob("runs/*.jsonl")):
    rows = [json.loads(l) for l in open(path)]
    last = rows[-1]
    print(
        f"{os.path.basename(path):60s} acc {last['global_acc']:.3f}  f1 {last['macro_f1']:.3f}  worst site {last['site_acc_spread']['min']:.3f}"
    )

In [ ]:
import shutil

shutil.make_archive("runs", "zip", "runs")
print("runs.zip written in", os.getcwd())

# Compact copy of every round: [round, acc, macro_f1, worst_site_acc, mean_site_acc, bytes_up]
for path in sorted(glob.glob("runs/*.jsonl")):
    rows = [json.loads(line) for line in open(path)]
    rounds = [r for r in rows if r["type"] == "round"]
    compact = [
        [
            r["round"],
            round(r["global_acc"], 4),
            round(r["macro_f1"], 4),
            round(r["site_acc_spread"]["min"], 4),
            round(r["site_acc_spread"]["mean"], 4),
            r["bytes_up"],
        ]
        for r in rounds
    ]
    print("RUN", os.path.basename(path), json.dumps(compact))
